# Base 2 — Clasificación: Hotel Booking Reservation
## Comparación experimental de Árbol de Decisión, Random Forest y KNN

**Variable objetivo:** `is_canceled`  
**Clases:** `0 = no cancelada`, `1 = cancelada`.

Este notebook conserva la estructura del trabajo anterior, pero reorganiza la metodología para que la selección de modelos sea válida:

- Se mantiene una partición **TRAIN 70 % / TEST 30 %** estratificada.
- **TEST no se utiliza para elegir hiperparámetros**.
- Dentro de TRAIN se utiliza **Stratified 5-Fold Cross-Validation**.
- Se comparan Árbol de Decisión, Random Forest y KNN por separado, pero dentro del mismo experimento.
- Se reportan Accuracy, Balanced Accuracy, Precision, Recall, Specificity, F1, ROC-AUC y PR-AUC.
- Se compara el Árbol antes y después de la poda.
- Se compara Random Forest sin regularización contra Random Forest ajustado por CV.
- Se evalúa `class_weight` en Árbol y Random Forest.
- El umbral de decisión se ajusta con predicciones OOF de TRAIN, nunca con TEST.
- Se generan matrices de confusión absolutas y normalizadas, curvas ROC y Precision–Recall.
- Las figuras y tablas se guardan en `resultados_clasificacion_cv/`.

> **Importante:** edita únicamente `DATA_PATH` si el CSV está en otra carpeta.

## 1. Carga, configuración y caracterización del dataset

In [ ]:
from pathlib import Path
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display
from matplotlib.ticker import PercentFormatter

from sklearn.base import clone
from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    GridSearchCV,
    cross_validate,
    cross_val_predict
)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier, NearestNeighbors
from sklearn.dummy import DummyClassifier
from sklearn.decomposition import PCA
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_curve,
    precision_recall_curve,
    make_scorer
)

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
N_SPLITS = 5

RESULTS_DIR = Path("resultados_clasificacion_cv")
RESULTS_DIR.mkdir(exist_ok=True)

# ------------------------------------------------------------
# CAMBIA SOLO ESTA LÍNEA SI EL CSV ESTÁ EN OTRA RUTA
# ------------------------------------------------------------
DATA_PATH = r"hotel_bookings_updated_2024-selected-columns.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset cargado correctamente")
print("Dimensiones:", df.shape)
display(df.head())

In [ ]:
resumen = pd.DataFrame({
    "Tipo": df.dtypes.astype(str),
    "Nulos": df.isna().sum(),
    "Nulos_%": (df.isna().mean() * 100).round(2),
    "Valores_unicos": df.nunique()
})

display(resumen)
print("Duplicados respecto a las columnas seleccionadas:", df.duplicated().sum())

### Interpretación

La base utilizada contiene una selección de columnas del dataset de reservas hoteleras.  
`arrival_date_year` debe revisarse porque, si solo contiene 2024, es una variable constante y no tiene capacidad discriminativa.

Los duplicados no se eliminan automáticamente: al trabajar con una selección reducida de atributos, dos reservas distintas pueden coincidir en todas las columnas disponibles.

### 1.1 Distribución de la variable objetivo

In [ ]:
class_table = (
    df["is_canceled"]
    .value_counts()
    .sort_index()
    .rename_axis("Clase")
    .to_frame("Cantidad")
)

class_table["Porcentaje"] = class_table["Cantidad"] / len(df)

display(
    class_table.assign(
        Porcentaje=(class_table["Porcentaje"] * 100).round(2)
    )
)

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(
    ["No cancelada (0)", "Cancelada (1)"],
    class_table["Porcentaje"]
)
ax.yaxis.set_major_formatter(PercentFormatter(1.0))
ax.set_ylabel("Porcentaje de reservas")
ax.set_title("Distribución de clases")
ax.grid(axis="y", alpha=0.25)

plt.tight_layout()
plt.savefig(RESULTS_DIR / "fig_01_distribucion_clases.png", dpi=160)
plt.show()

### Lectura de la distribución

Una proporción cercana a 63/37 representa una **asimetría moderada**, no un desbalance extremo.

Por esa razón:

1. se utiliza `stratify=y` en TRAIN/TEST;
2. se utiliza `StratifiedKFold` dentro de TRAIN;
3. no se evalúa únicamente Accuracy;
4. se comparan `class_weight=None` y `class_weight="balanced"`/`"balanced_subsample"`;
5. no se introduce SMOTE de entrada, porque primero se evalúan técnicas menos invasivas.

### 1.2 Revisión de valores atípicos

In [ ]:
numeric_eda = df.select_dtypes(include="number").columns.drop("is_canceled").tolist()

outlier_rows = []

for col in numeric_eda:
    q1, q3 = df[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    low = q1 - 1.5 * iqr
    high = q3 + 1.5 * iqr
    n_out = ((df[col] < low) | (df[col] > high)).sum()

    outlier_rows.append({
        "Variable": col,
        "Q1": q1,
        "Q3": q3,
        "Limite_inferior": low,
        "Limite_superior": high,
        "Casos_IQR": n_out,
        "Min": df[col].min(),
        "Max": df[col].max()
    })

outlier_table = pd.DataFrame(outlier_rows)
display(outlier_table)

La regla IQR se usa aquí como **diagnóstico**, no como criterio automático de eliminación.  
En variables discretas como `adults` o noches de estancia, un valor poco frecuente no necesariamente es un error.

## 2. Preparación de datos y prevención de data leakage

In [ ]:
# Se elimina la variable objetivo y la variable constante.
drop_cols = ["is_canceled"]

if "arrival_date_year" in df.columns and df["arrival_date_year"].nunique() <= 1:
    drop_cols.append("arrival_date_year")

X = df.drop(columns=drop_cols).copy()
y = df["is_canceled"].astype(int).copy()

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=RANDOM_STATE,
    stratify=y
)

numeric_cols = X_train.select_dtypes(include="number").columns.tolist()
categorical_cols = X_train.select_dtypes(exclude="number").columns.tolist()

print(f"TRAIN: {len(X_train):,} ({len(X_train)/len(X):.0%})")
print(f"TEST : {len(X_test):,} ({len(X_test)/len(X):.0%})")

print("\nDistribución TRAIN:")
display((y_train.value_counts(normalize=True).sort_index() * 100).round(2).to_frame("%"))

print("Distribución TEST:")
display((y_test.value_counts(normalize=True).sort_index() * 100).round(2).to_frame("%"))

print("Variables numéricas:", numeric_cols)
print("Variables categóricas:", categorical_cols)

In [ ]:
def make_ohe():
    # Compatibilidad con distintas versiones de scikit-learn.
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)

pre_tree = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), numeric_cols),
    ("cat", make_ohe(), categorical_cols)
])

pre_knn = ColumnTransformer([
    (
        "num",
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]),
        numeric_cols
    ),
    ("cat", make_ohe(), categorical_cols)
])

cv = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE
)

### ¿Por qué dos preprocesamientos?

- **Árbol y Random Forest:** no necesitan estandarización porque sus divisiones dependen de umbrales.
- **KNN:** sí necesita escalado porque usa distancias.
- Todas las transformaciones permanecen dentro de `Pipeline`, por lo que en cada fold el preprocesamiento se aprende únicamente con la parte de entrenamiento del fold.

## 2.1 Métricas comunes del experimento

In [ ]:
def specificity_score(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    return tn / (tn + fp) if (tn + fp) else 0.0

specificity_scorer = make_scorer(specificity_score)

SCORING = {
    "accuracy": "accuracy",
    "balanced_accuracy": "balanced_accuracy",
    "precision": "precision",
    "recall": "recall",
    "specificity": specificity_scorer,
    "f1": "f1",
    "roc_auc": "roc_auc",
    "pr_auc": "average_precision"
}

PRIMARY_METRIC = "f1"

def classification_metrics(y_true, y_pred, y_prob=None):
    result = {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Balanced_Accuracy": balanced_accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "Specificity": specificity_score(y_true, y_pred),
        "F1": f1_score(y_true, y_pred, zero_division=0)
    }

    if y_prob is not None:
        result["ROC_AUC"] = roc_auc_score(y_true, y_prob)
        result["PR_AUC"] = average_precision_score(y_true, y_prob)

    return result

def cv_summary(cv_result, model_name):
    row = {"Modelo": model_name}

    for metric in SCORING:
        values = cv_result[f"test_{metric}"]
        row[f"{metric}_mean"] = values.mean()
        row[f"{metric}_std"] = values.std()

    row["fit_time_mean_s"] = cv_result["fit_time"].mean()
    row["score_time_mean_s"] = cv_result["score_time"].mean()

    return row

**Métrica primaria de selección:** `F1` de la clase `1 = cancelada`.

Se utiliza F1 porque no existe todavía un costo empresarial explícito para falsos positivos frente a falsos negativos. F1 obliga a equilibrar Precision y Recall.  
Balanced Accuracy, ROC-AUC y PR-AUC se reportan como métricas complementarias.

## 2.2 Baseline — modelo que siempre predice la clase mayoritaria

In [ ]:
baseline = Pipeline([
    ("preprocessor", pre_tree),
    ("model", DummyClassifier(strategy="most_frequent"))
])

baseline.fit(X_train, y_train)

baseline_pred = baseline.predict(X_test)
baseline_prob = baseline.predict_proba(X_test)[:, 1]

baseline_metrics = classification_metrics(
    y_test,
    baseline_pred,
    baseline_prob
)

display(
    pd.DataFrame([baseline_metrics], index=["Baseline"]).T
)

print(
    "\nEl baseline sirve para comprobar cuánto mejora realmente "
    "cada modelo respecto a predecir siempre la clase mayoritaria."
)

# 3. Árbol de Decisión

## 3.1 Cross-Validation de atributos candidatos al nodo raíz

Para estudiar la capacidad discriminativa individual de cada atributo se entrena un **Decision Stump** (`max_depth=1`) usando únicamente ese atributo.

Esto no significa que el atributo ganador deba ser obligatoriamente la raíz del árbol completo. La segunda tabla registra qué característica eligió realmente CART como raíz en cada fold.

In [ ]:
root_candidate_rows = []

for col in X_train.columns:
    if col in numeric_cols:
        pre_one = ColumnTransformer([
            ("num", SimpleImputer(strategy="median"), [col])
        ])
    else:
        pre_one = ColumnTransformer([
            ("cat", make_ohe(), [col])
        ])

    stump = Pipeline([
        ("preprocessor", pre_one),
        ("model", DecisionTreeClassifier(
            max_depth=1,
            class_weight="balanced",
            random_state=RANDOM_STATE
        ))
    ])

    scores = cross_validate(
        stump,
        X_train[[col]],
        y_train,
        cv=cv,
        scoring=SCORING,
        n_jobs=1
    )

    root_candidate_rows.append({
        "Atributo": col,
        "Balanced_Accuracy_CV": scores["test_balanced_accuracy"].mean(),
        "F1_CV": scores["test_f1"].mean(),
        "Recall_CV": scores["test_recall"].mean(),
        "ROC_AUC_CV": scores["test_roc_auc"].mean(),
        "PR_AUC_CV": scores["test_pr_auc"].mean()
    })

root_candidate_results = (
    pd.DataFrame(root_candidate_rows)
    .sort_values(["F1_CV", "Balanced_Accuracy_CV"], ascending=False)
    .reset_index(drop=True)
)

display(root_candidate_results.round(4))
root_candidate_results.to_csv(
    RESULTS_DIR / "tabla_01_cv_atributos_raiz.csv",
    index=False
)

fig, ax = plt.subplots(figsize=(9, 5))
plot_data = root_candidate_results.sort_values("F1_CV")
ax.barh(plot_data["Atributo"], plot_data["F1_CV"])
ax.set_xlabel("F1 medio — 5-Fold CV")
ax.set_title("Capacidad individual de cada atributo como Decision Stump")
ax.grid(axis="x", alpha=0.25)

plt.tight_layout()
plt.savefig(RESULTS_DIR / "fig_02_cv_atributos_raiz.png", dpi=160)
plt.show()

### 3.2 Nodo raíz elegido realmente por CART en cada fold

In [ ]:
root_fold_rows = []

for fold, (tr_idx, va_idx) in enumerate(cv.split(X_train, y_train), start=1):
    X_tr = X_train.iloc[tr_idx]
    X_va = X_train.iloc[va_idx]
    y_tr = y_train.iloc[tr_idx]
    y_va = y_train.iloc[va_idx]

    pipe = Pipeline([
        ("preprocessor", clone(pre_tree)),
        ("model", DecisionTreeClassifier(
            max_depth=3,
            class_weight="balanced",
            random_state=RANDOM_STATE
        ))
    ])

    pipe.fit(X_tr, y_tr)

    pre = pipe.named_steps["preprocessor"]
    est = pipe.named_steps["model"]
    feature_names = pre.get_feature_names_out()

    root_idx = est.tree_.feature[0]
    root_feature = feature_names[root_idx] if root_idx >= 0 else "Sin división"
    root_threshold = est.tree_.threshold[0] if root_idx >= 0 else np.nan

    pred = pipe.predict(X_va)
    prob = pipe.predict_proba(X_va)[:, 1]
    m = classification_metrics(y_va, pred, prob)

    root_fold_rows.append({
        "Fold": fold,
        "Raiz": root_feature,
        "Umbral": root_threshold,
        "Balanced_Accuracy": m["Balanced_Accuracy"],
        "F1": m["F1"],
        "Recall": m["Recall"],
        "ROC_AUC": m["ROC_AUC"]
    })

root_fold_results = pd.DataFrame(root_fold_rows)

display(root_fold_results.round(4))
root_fold_results.to_csv(
    RESULTS_DIR / "tabla_02_raiz_real_por_fold.csv",
    index=False
)

print("\nFrecuencia de cada raíz:")
display(root_fold_results["Raiz"].value_counts().to_frame("Frecuencia"))

## 3.3 Pre-poda del Árbol mediante Stratified 5-Fold CV

In [ ]:
# Se usa una muestra estratificada de TRAIN para que la búsqueda sea reproducible
# y ejecutable en un portátil. TEST continúa completamente aislado.

TREE_TUNING_SAMPLE = min(30000, len(X_train))

if TREE_TUNING_SAMPLE < len(X_train):
    X_tree_tune, _, y_tree_tune, _ = train_test_split(
        X_train,
        y_train,
        train_size=TREE_TUNING_SAMPLE,
        random_state=RANDOM_STATE,
        stratify=y_train
    )
else:
    X_tree_tune, y_tree_tune = X_train.copy(), y_train.copy()

tree_pipe = Pipeline([
    ("preprocessor", clone(pre_tree)),
    ("model", DecisionTreeClassifier(random_state=RANDOM_STATE))
])

tree_param_grid = {
    "model__max_depth": [3, 4, 5, 6, 8, 10, 12, None],
    "model__min_samples_leaf": [1, 10, 25, 50, 100],
    "model__class_weight": [None, "balanced"]
}

tree_search = GridSearchCV(
    estimator=tree_pipe,
    param_grid=tree_param_grid,
    scoring=SCORING,
    refit=PRIMARY_METRIC,
    cv=cv,
    n_jobs=-1,
    return_train_score=True
)

t0 = time.perf_counter()
tree_search.fit(X_tree_tune, y_tree_tune)
tree_search_time = time.perf_counter() - t0

print("Mejores parámetros de pre-poda:")
print(tree_search.best_params_)
print(f"Mejor F1 CV: {tree_search.best_score_:.4f}")
print(f"Tiempo búsqueda: {tree_search_time:.2f} s")

tree_cv = pd.DataFrame(tree_search.cv_results_)
tree_cv.to_csv(
    RESULTS_DIR / "tabla_03_tree_prepoda_cv_completa.csv",
    index=False
)

tree_top = tree_cv.sort_values(
    "rank_test_f1"
)[[
    "param_model__max_depth",
    "param_model__min_samples_leaf",
    "param_model__class_weight",
    "mean_train_f1",
    "mean_test_f1",
    "std_test_f1",
    "mean_test_balanced_accuracy",
    "mean_test_recall",
    "mean_test_roc_auc",
    "mean_test_pr_auc"
]].head(20)

display(tree_top.round(4))

## 3.4 Poda Cost-Complexity (`ccp_alpha`) por Cross-Validation

In [ ]:
best_depth = tree_search.best_params_["model__max_depth"]
best_leaf = tree_search.best_params_["model__min_samples_leaf"]
best_weight = tree_search.best_params_["model__class_weight"]

# Ajustamos el preprocesador sobre la muestra de tuning para obtener
# una ruta razonable de alphas sin utilizar TEST.
pre_alpha = clone(pre_tree)
X_alpha = pre_alpha.fit_transform(X_tree_tune, y_tree_tune)

alpha_tree = DecisionTreeClassifier(
    max_depth=best_depth,
    min_samples_leaf=best_leaf,
    class_weight=best_weight,
    random_state=RANDOM_STATE
)

path = alpha_tree.cost_complexity_pruning_path(X_alpha, y_tree_tune)
positive_alphas = np.unique(path.ccp_alphas[path.ccp_alphas >= 0])

if len(positive_alphas) > 12:
    idx = np.linspace(0, len(positive_alphas) - 1, 12).astype(int)
    alpha_candidates = np.unique(positive_alphas[idx])
else:
    alpha_candidates = positive_alphas

# Siempre incluir alpha = 0
alpha_candidates = np.unique(np.r_[0.0, alpha_candidates])

tree_alpha_pipe = Pipeline([
    ("preprocessor", clone(pre_tree)),
    ("model", DecisionTreeClassifier(
        max_depth=best_depth,
        min_samples_leaf=best_leaf,
        class_weight=best_weight,
        random_state=RANDOM_STATE
    ))
])

alpha_search = GridSearchCV(
    tree_alpha_pipe,
    param_grid={"model__ccp_alpha": alpha_candidates.tolist()},
    scoring=SCORING,
    refit=PRIMARY_METRIC,
    cv=cv,
    n_jobs=-1,
    return_train_score=True
)

alpha_search.fit(X_tree_tune, y_tree_tune)

best_tree_alpha = alpha_search.best_params_["model__ccp_alpha"]

print("ccp_alpha seleccionado por CV:", best_tree_alpha)
print("F1 CV:", round(alpha_search.best_score_, 4))

tree_alpha_cv = pd.DataFrame(alpha_search.cv_results_).sort_values(
    "param_model__ccp_alpha"
)

tree_alpha_cv.to_csv(
    RESULTS_DIR / "tabla_04_tree_ccp_alpha_cv.csv",
    index=False
)

display(
    tree_alpha_cv[[
        "param_model__ccp_alpha",
        "mean_train_f1",
        "mean_test_f1",
        "std_test_f1",
        "mean_test_balanced_accuracy",
        "mean_test_recall"
    ]].round(4)
)

fig, ax = plt.subplots(figsize=(8, 4.8))
ax.plot(
    tree_alpha_cv["param_model__ccp_alpha"].astype(float),
    tree_alpha_cv["mean_test_f1"],
    marker="o",
    label="F1 validación"
)
ax.plot(
    tree_alpha_cv["param_model__ccp_alpha"].astype(float),
    tree_alpha_cv["mean_train_f1"],
    marker="o",
    label="F1 entrenamiento"
)
ax.axvline(
    best_tree_alpha,
    linestyle="--",
    label=f"alpha elegido={best_tree_alpha:.6g}"
)
ax.set_xlabel("ccp_alpha")
ax.set_ylabel("F1")
ax.set_title("Árbol — poda Cost-Complexity por 5-Fold CV")
ax.legend()
ax.grid(alpha=0.25)

plt.tight_layout()
plt.savefig(RESULTS_DIR / "fig_03_tree_ccp_alpha_cv.png", dpi=160)
plt.show()

## 3.5 Árbol sin poda vs árbol podado

In [ ]:
tree_unpruned = Pipeline([
    ("preprocessor", clone(pre_tree)),
    ("model", DecisionTreeClassifier(
        random_state=RANDOM_STATE
    ))
])

tree_pruned = Pipeline([
    ("preprocessor", clone(pre_tree)),
    ("model", DecisionTreeClassifier(
        max_depth=best_depth,
        min_samples_leaf=best_leaf,
        class_weight=best_weight,
        ccp_alpha=best_tree_alpha,
        random_state=RANDOM_STATE
    ))
])

tree_compare_rows = []

for name, model in [
    ("Árbol sin poda", tree_unpruned),
    ("Árbol podado CV", tree_pruned)
]:
    scores = cross_validate(
        model,
        X_tree_tune,
        y_tree_tune,
        cv=cv,
        scoring=SCORING,
        n_jobs=-1,
        return_train_score=True
    )

    model.fit(X_train, y_train)
    est = model.named_steps["model"]

    tree_compare_rows.append({
        "Modelo": name,
        "F1_CV_mean": scores["test_f1"].mean(),
        "F1_CV_std": scores["test_f1"].std(),
        "Balanced_Accuracy_CV": scores["test_balanced_accuracy"].mean(),
        "Recall_CV": scores["test_recall"].mean(),
        "Profundidad": est.get_depth(),
        "Nodos": est.tree_.node_count,
        "Hojas": est.get_n_leaves()
    })

tree_compare_cv = pd.DataFrame(tree_compare_rows)
display(tree_compare_cv.round(4))

tree_compare_cv.to_csv(
    RESULTS_DIR / "tabla_05_tree_antes_despues_poda_cv.csv",
    index=False
)

## 3.6 Imagen del Árbol final

In [ ]:
tree_pruned.fit(X_train, y_train)

tree_feat = tree_pruned.named_steps["preprocessor"].get_feature_names_out()
tree_est = tree_pruned.named_steps["model"]

fig, ax = plt.subplots(figsize=(20, 9))
plot_tree(
    tree_est,
    feature_names=tree_feat,
    class_names=["No cancelada", "Cancelada"],
    filled=False,
    max_depth=3,
    fontsize=7,
    ax=ax
)
ax.set_title(
    f"Árbol final podado — primeros 3 niveles "
    f"(profundidad real={tree_est.get_depth()}, hojas={tree_est.get_n_leaves()})"
)

plt.tight_layout()
plt.savefig(RESULTS_DIR / "fig_04_arbol_final.png", dpi=170)
plt.show()

# 4. Random Forest Classifier

La selección se realiza por etapas para evitar una búsqueda combinatoria excesiva:

1. estructura (`max_depth`, `min_samples_leaf`, `class_weight`);
2. número de árboles y `max_features`;
3. poda mediante `ccp_alpha`.

La muestra de tuning proviene exclusivamente de TRAIN.

In [ ]:
RF_TUNING_SAMPLE = min(15000, len(X_train))

if RF_TUNING_SAMPLE < len(X_train):
    X_rf_tune, _, y_rf_tune, _ = train_test_split(
        X_train,
        y_train,
        train_size=RF_TUNING_SAMPLE,
        random_state=RANDOM_STATE,
        stratify=y_train
    )
else:
    X_rf_tune, y_rf_tune = X_train.copy(), y_train.copy()

rf_structure_pipe = Pipeline([
    ("preprocessor", clone(pre_tree)),
    ("model", RandomForestClassifier(
        n_estimators=50,
        random_state=RANDOM_STATE,
        n_jobs=-1
    ))
])

rf_structure_grid = {
    "model__max_depth": [6, 10, 14, None],
    "model__min_samples_leaf": [1, 10, 50],
    "model__class_weight": [None, "balanced_subsample"]
}

rf_structure_search = GridSearchCV(
    rf_structure_pipe,
    rf_structure_grid,
    scoring=SCORING,
    refit=PRIMARY_METRIC,
    cv=cv,
    n_jobs=1,
    return_train_score=True
)

rf_structure_search.fit(X_rf_tune, y_rf_tune)

rf_best_depth = rf_structure_search.best_params_["model__max_depth"]
rf_best_leaf = rf_structure_search.best_params_["model__min_samples_leaf"]
rf_best_weight = rf_structure_search.best_params_["model__class_weight"]

print("Mejor estructura RF:")
print(rf_structure_search.best_params_)
print("F1 CV:", round(rf_structure_search.best_score_, 4))

## 4.1 Número de árboles y `max_features` mediante CV

In [ ]:
rf_n_pipe = Pipeline([
    ("preprocessor", clone(pre_tree)),
    ("model", RandomForestClassifier(
        max_depth=rf_best_depth,
        min_samples_leaf=rf_best_leaf,
        class_weight=rf_best_weight,
        random_state=RANDOM_STATE,
        n_jobs=-1
    ))
])

# Puedes añadir 2000 si deseas ampliar el experimento.
rf_n_grid = {
    "model__n_estimators": [50, 100, 250, 500, 1000],
    "model__max_features": ["sqrt", 0.5, 1.0]
}

rf_n_search = GridSearchCV(
    rf_n_pipe,
    rf_n_grid,
    scoring=SCORING,
    refit=PRIMARY_METRIC,
    cv=cv,
    n_jobs=1,
    return_train_score=True
)

t0 = time.perf_counter()
rf_n_search.fit(X_rf_tune, y_rf_tune)
rf_search_time = time.perf_counter() - t0

rf_best_n = rf_n_search.best_params_["model__n_estimators"]
rf_best_features = rf_n_search.best_params_["model__max_features"]

print("Configuración elegida:")
print(rf_n_search.best_params_)
print("F1 CV:", round(rf_n_search.best_score_, 4))
print(f"Tiempo búsqueda: {rf_search_time:.2f} s")

rf_n_cv = pd.DataFrame(rf_n_search.cv_results_)
rf_n_cv.to_csv(
    RESULTS_DIR / "tabla_06_rf_n_estimators_cv.csv",
    index=False
)

display(
    rf_n_cv.sort_values("rank_test_f1")[[
        "param_model__n_estimators",
        "param_model__max_features",
        "mean_train_f1",
        "mean_test_f1",
        "std_test_f1",
        "mean_test_balanced_accuracy",
        "mean_test_recall",
        "mean_test_roc_auc"
    ]].head(20).round(4)
)

# Gráfica para observar estabilización según cantidad de árboles
plot_rf = (
    rf_n_cv.groupby("param_model__n_estimators", as_index=False)
    .agg(
        F1_CV=("mean_test_f1", "max"),
        Tiempo_fit=("mean_fit_time", "mean")
    )
    .sort_values("param_model__n_estimators")
)

fig, ax = plt.subplots(figsize=(7.5, 4.6))
ax.plot(
    plot_rf["param_model__n_estimators"].astype(int),
    plot_rf["F1_CV"],
    marker="o"
)
ax.set_xlabel("Número de árboles")
ax.set_ylabel("Mejor F1 CV")
ax.set_title("Random Forest — rendimiento vs número de árboles")
ax.grid(alpha=0.25)

plt.tight_layout()
plt.savefig(RESULTS_DIR / "fig_05_rf_arboles_f1.png", dpi=160)
plt.show()

fig, ax = plt.subplots(figsize=(7.5, 4.6))
ax.plot(
    plot_rf["param_model__n_estimators"].astype(int),
    plot_rf["Tiempo_fit"],
    marker="o"
)
ax.set_xlabel("Número de árboles")
ax.set_ylabel("Tiempo medio de ajuste por fold (s)")
ax.set_title("Random Forest — costo computacional")
ax.grid(alpha=0.25)

plt.tight_layout()
plt.savefig(RESULTS_DIR / "fig_06_rf_arboles_tiempo.png", dpi=160)
plt.show()

## 4.2 `ccp_alpha` para Random Forest

In [ ]:
rf_alpha_pipe = Pipeline([
    ("preprocessor", clone(pre_tree)),
    ("model", RandomForestClassifier(
        n_estimators=rf_best_n,
        max_depth=rf_best_depth,
        min_samples_leaf=rf_best_leaf,
        max_features=rf_best_features,
        class_weight=rf_best_weight,
        random_state=RANDOM_STATE,
        n_jobs=-1
    ))
])

RF_ALPHA_VALUES = [0.0, 1e-5, 5e-5, 1e-4, 5e-4]

rf_alpha_search = GridSearchCV(
    rf_alpha_pipe,
    param_grid={"model__ccp_alpha": RF_ALPHA_VALUES},
    scoring=SCORING,
    refit=PRIMARY_METRIC,
    cv=cv,
    n_jobs=1,
    return_train_score=True
)

rf_alpha_search.fit(X_rf_tune, y_rf_tune)

rf_best_alpha = rf_alpha_search.best_params_["model__ccp_alpha"]

print("ccp_alpha RF seleccionado:", rf_best_alpha)
print("F1 CV:", round(rf_alpha_search.best_score_, 4))

rf_alpha_cv = pd.DataFrame(rf_alpha_search.cv_results_)
rf_alpha_cv.to_csv(
    RESULTS_DIR / "tabla_07_rf_ccp_alpha_cv.csv",
    index=False
)

display(
    rf_alpha_cv[[
        "param_model__ccp_alpha",
        "mean_train_f1",
        "mean_test_f1",
        "std_test_f1",
        "mean_test_balanced_accuracy",
        "mean_test_recall"
    ]].round(4)
)

## 4.3 Random Forest sin regularización vs ajustado por CV

In [ ]:
rf_unregularized = Pipeline([
    ("preprocessor", clone(pre_tree)),
    ("model", RandomForestClassifier(
        n_estimators=100,
        random_state=RANDOM_STATE,
        n_jobs=-1
    ))
])

rf_tuned = Pipeline([
    ("preprocessor", clone(pre_tree)),
    ("model", RandomForestClassifier(
        n_estimators=rf_best_n,
        max_depth=rf_best_depth,
        min_samples_leaf=rf_best_leaf,
        max_features=rf_best_features,
        class_weight=rf_best_weight,
        ccp_alpha=rf_best_alpha,
        oob_score=True,
        bootstrap=True,
        random_state=RANDOM_STATE,
        n_jobs=-1
    ))
])

rf_compare_rows = []

for name, model in [
    ("RF sin regularización", rf_unregularized),
    ("RF ajustado CV", rf_tuned)
]:
    scores = cross_validate(
        model,
        X_rf_tune,
        y_rf_tune,
        cv=cv,
        scoring=SCORING,
        n_jobs=1,
        return_train_score=True
    )

    rf_compare_rows.append({
        "Modelo": name,
        "F1_train": scores["train_f1"].mean(),
        "F1_CV": scores["test_f1"].mean(),
        "F1_CV_std": scores["test_f1"].std(),
        "Balanced_Accuracy_CV": scores["test_balanced_accuracy"].mean(),
        "Recall_CV": scores["test_recall"].mean(),
        "ROC_AUC_CV": scores["test_roc_auc"].mean()
    })

rf_compare_cv = pd.DataFrame(rf_compare_rows)
display(rf_compare_cv.round(4))

rf_compare_cv.to_csv(
    RESULTS_DIR / "tabla_08_rf_antes_despues_cv.csv",
    index=False
)

## 4.4 Imagen de un árbol representativo del Random Forest final

In [ ]:
rf_tuned.fit(X_train, y_train)

rf_pre = rf_tuned.named_steps["preprocessor"]
rf_est = rf_tuned.named_steps["model"]
rf_feat = rf_pre.get_feature_names_out()

print("OOB score del bosque final:", round(rf_est.oob_score_, 4))

representative_tree = rf_est.estimators_[0]

fig, ax = plt.subplots(figsize=(20, 9))
plot_tree(
    representative_tree,
    feature_names=rf_feat,
    class_names=["No cancelada", "Cancelada"],
    filled=False,
    max_depth=3,
    fontsize=7,
    ax=ax
)
ax.set_title(
    "Random Forest final — árbol representativo (primeros 3 niveles)"
)

plt.tight_layout()
plt.savefig(RESULTS_DIR / "fig_07_rf_arbol_representativo.png", dpi=170)
plt.show()

## 4.5 Importancia de características en Random Forest

In [ ]:
rf_importance = pd.DataFrame({
    "Caracteristica": rf_feat,
    "Importancia_MDI": rf_est.feature_importances_
}).sort_values("Importancia_MDI", ascending=False)

display(rf_importance.head(20).round(5))

fig, ax = plt.subplots(figsize=(9, 6))
top_imp = rf_importance.head(15).sort_values("Importancia_MDI")
ax.barh(top_imp["Caracteristica"], top_imp["Importancia_MDI"])
ax.set_xlabel("Importancia MDI")
ax.set_title("Random Forest — 15 características con mayor importancia")
ax.grid(axis="x", alpha=0.25)

plt.tight_layout()
plt.savefig(RESULTS_DIR / "fig_08_rf_importancias.png", dpi=160)
plt.show()

rf_importance.to_csv(
    RESULTS_DIR / "tabla_09_rf_importancias.csv",
    index=False
)

Estas importancias describen contribuciones internas del bosque. **No implican causalidad**.  
La importancia MDI también puede favorecer características continuas o con muchas posibilidades de división, por lo que debe interpretarse como diagnóstico del modelo.

# 5. KNN Classifier

KNN utiliza un Pipeline diferente porque sus distancias requieren escalado de variables numéricas.

Se seleccionan conjuntamente:

- `n_neighbors` (K),
- `weights`,
- `p=1` Manhattan o `p=2` Euclidiana.

La selección se realiza con Stratified 5-Fold CV y F1 como métrica primaria.

In [ ]:
KNN_TUNING_SAMPLE = min(20000, len(X_train))

if KNN_TUNING_SAMPLE < len(X_train):
    X_knn_tune, _, y_knn_tune, _ = train_test_split(
        X_train,
        y_train,
        train_size=KNN_TUNING_SAMPLE,
        random_state=RANDOM_STATE,
        stratify=y_train
    )
else:
    X_knn_tune, y_knn_tune = X_train.copy(), y_train.copy()

knn_pipe = Pipeline([
    ("preprocessor", clone(pre_knn)),
    ("model", KNeighborsClassifier(n_jobs=-1))
])

knn_grid = {
    "model__n_neighbors": [3, 5, 7, 10, 15, 20, 30, 50, 75, 100, 150],
    "model__weights": ["uniform", "distance"],
    "model__p": [1, 2]
}

knn_search = GridSearchCV(
    knn_pipe,
    knn_grid,
    scoring=SCORING,
    refit=PRIMARY_METRIC,
    cv=cv,
    n_jobs=1,
    return_train_score=False
)

t0 = time.perf_counter()
knn_search.fit(X_knn_tune, y_knn_tune)
knn_search_time = time.perf_counter() - t0

print("Mejor configuración KNN:")
print(knn_search.best_params_)
print("F1 CV:", round(knn_search.best_score_, 4))
print(f"Tiempo búsqueda: {knn_search_time:.2f} s")

knn_cv = pd.DataFrame(knn_search.cv_results_)
knn_cv.to_csv(
    RESULTS_DIR / "tabla_10_knn_cv.csv",
    index=False
)

display(
    knn_cv.sort_values("rank_test_f1")[[
        "param_model__n_neighbors",
        "param_model__weights",
        "param_model__p",
        "mean_test_f1",
        "std_test_f1",
        "mean_test_balanced_accuracy",
        "mean_test_recall",
        "mean_test_roc_auc",
        "mean_test_pr_auc"
    ]].head(20).round(4)
)

best_p = knn_search.best_params_["model__p"]

fig, ax = plt.subplots(figsize=(9, 5))

for weight in ["uniform", "distance"]:
    temp = knn_cv[
        (knn_cv["param_model__weights"] == weight) &
        (knn_cv["param_model__p"].astype(int) == int(best_p))
    ].sort_values("param_model__n_neighbors")

    ax.plot(
        temp["param_model__n_neighbors"].astype(int),
        temp["mean_test_f1"],
        marker="o",
        label=weight
    )

best_k = knn_search.best_params_["model__n_neighbors"]

ax.axvline(
    best_k,
    linestyle="--",
    label=f"K elegido={best_k}"
)

ax.set_xlabel("K")
ax.set_ylabel("F1 medio — 5-Fold CV")
ax.set_title(
    f"KNN — selección de K y pesos por Cross-Validation (p={best_p})"
)
ax.legend()
ax.grid(alpha=0.25)

plt.tight_layout()
plt.savefig(RESULTS_DIR / "fig_09_knn_cv.png", dpi=160)
plt.show()

## 5.1 Modelo KNN final

In [ ]:
knn_final = clone(knn_search.best_estimator_)
knn_final.fit(X_train, y_train)

print("KNN final:")
print(knn_final.named_steps["model"])

## 5.2 Visualización del vecindario KNN

In [ ]:
# KNN no genera clústeres. Esta figura muestra los vecinos reales
# de un punto de TEST y utiliza PCA únicamente para proyectarlos a 2D.

knn_pre = knn_final.named_steps["preprocessor"]
knn_est = knn_final.named_steps["model"]

# Submuestra de TRAIN para que la figura sea legible.
VIS_SAMPLE = min(5000, len(X_train))
X_vis, _, y_vis, _ = train_test_split(
    X_train,
    y_train,
    train_size=VIS_SAMPLE,
    random_state=RANDOM_STATE,
    stratify=y_train
)

X_vis_t = knn_pre.transform(X_vis)

# Seleccionamos una reserva cancelada del TEST si existe.
positive_positions = np.flatnonzero(y_test.to_numpy() == 1)
point_pos = int(positive_positions[0]) if len(positive_positions) else 0

X_point = X_test.iloc[[point_pos]]
X_point_t = knn_pre.transform(X_point)

n_neighbors_vis = min(
    knn_est.n_neighbors,
    len(X_vis_t)
)

nn = NearestNeighbors(
    n_neighbors=n_neighbors_vis,
    metric="minkowski",
    p=knn_est.p
)
nn.fit(X_vis_t)

distances, indices = nn.kneighbors(X_point_t)

pca = PCA(n_components=2, random_state=RANDOM_STATE)
combined = np.vstack([X_vis_t, X_point_t])
proj = pca.fit_transform(combined)

train_proj = proj[:-1]
point_proj = proj[-1]
neighbor_idx = indices[0]

fig, ax = plt.subplots(figsize=(8, 6))

for cls, label in [(0, "No cancelada"), (1, "Cancelada")]:
    mask = y_vis.to_numpy() == cls
    ax.scatter(
        train_proj[mask, 0],
        train_proj[mask, 1],
        s=12,
        alpha=0.15,
        label=label
    )

ax.scatter(
    train_proj[neighbor_idx, 0],
    train_proj[neighbor_idx, 1],
    s=65,
    facecolors="none",
    edgecolors="black",
    linewidths=1.2,
    label=f"{n_neighbors_vis} vecinos"
)

ax.scatter(
    point_proj[0],
    point_proj[1],
    marker="*",
    s=220,
    label="Reserva consultada"
)

ax.set_title(
    "KNN — vecinos determinados en el espacio escalado\n"
    "y proyectados a 2D mediante PCA"
)
ax.set_xlabel("Componente principal 1")
ax.set_ylabel("Componente principal 2")
ax.legend()
ax.grid(alpha=0.2)

plt.tight_layout()
plt.savefig(RESULTS_DIR / "fig_10_knn_vecindario.png", dpi=160)
plt.show()

# 6. Comparación común por Cross-Validation

In [ ]:
# Se comparan las configuraciones finales con los mismos folds
# y el threshold estándar 0.5. Esto permite comparar estabilidad
# antes de estudiar el ajuste posterior del threshold.

CV_COMPARISON_SAMPLE = min(25000, len(X_train))

if CV_COMPARISON_SAMPLE < len(X_train):
    X_cv_cmp, _, y_cv_cmp, _ = train_test_split(
        X_train,
        y_train,
        train_size=CV_COMPARISON_SAMPLE,
        random_state=RANDOM_STATE,
        stratify=y_train
    )
else:
    X_cv_cmp, y_cv_cmp = X_train.copy(), y_train.copy()

common_models = {
    "Árbol podado": clone(tree_pruned),
    "Random Forest ajustado": clone(rf_tuned),
    "KNN ajustado": clone(knn_final)
}

common_cv_rows = []

for name, model in common_models.items():
    scores = cross_validate(
        model,
        X_cv_cmp,
        y_cv_cmp,
        cv=cv,
        scoring=SCORING,
        n_jobs=1
    )
    common_cv_rows.append(cv_summary(scores, name))

common_cv_results = pd.DataFrame(common_cv_rows)

display(common_cv_results.round(4))
common_cv_results.to_csv(
    RESULTS_DIR / "tabla_11_comparacion_modelos_cv.csv",
    index=False
)

# 7. Selección de threshold usando únicamente TRAIN

El threshold `0.5` no es obligatorio.

Para cada modelo final se generan probabilidades **Out-Of-Fold (OOF)** mediante Stratified 5-Fold CV sobre una muestra de TRAIN. Después se busca el threshold que maximiza F1.

TEST sigue sin intervenir.

In [ ]:
THRESHOLD_SAMPLE = min(25000, len(X_train))

if THRESHOLD_SAMPLE < len(X_train):
    X_thr, _, y_thr, _ = train_test_split(
        X_train,
        y_train,
        train_size=THRESHOLD_SAMPLE,
        random_state=RANDOM_STATE,
        stratify=y_train
    )
else:
    X_thr, y_thr = X_train.copy(), y_train.copy()

threshold_grid = np.arange(0.10, 0.91, 0.01)

threshold_rows = []
selected_thresholds = {}

for model_name, model in common_models.items():
    print("Calculando probabilidades OOF:", model_name)

    oof_prob = cross_val_predict(
        clone(model),
        X_thr,
        y_thr,
        cv=cv,
        method="predict_proba",
        n_jobs=1
    )[:, 1]

    model_rows = []

    for threshold in threshold_grid:
        pred = (oof_prob >= threshold).astype(int)
        row = {
            "Modelo": model_name,
            "Threshold": threshold,
            "Precision": precision_score(y_thr, pred, zero_division=0),
            "Recall": recall_score(y_thr, pred, zero_division=0),
            "F1": f1_score(y_thr, pred, zero_division=0),
            "Balanced_Accuracy": balanced_accuracy_score(y_thr, pred)
        }
        model_rows.append(row)

    model_df = pd.DataFrame(model_rows)

    best_row = model_df.loc[
        model_df["F1"].idxmax()
    ]

    selected_thresholds[model_name] = float(best_row["Threshold"])
    threshold_rows.extend(model_rows)

    print(
        f"  threshold elegido={best_row['Threshold']:.2f} | "
        f"F1 OOF={best_row['F1']:.4f} | "
        f"Recall OOF={best_row['Recall']:.4f}"
    )

threshold_results = pd.DataFrame(threshold_rows)

display(
    threshold_results.loc[
        threshold_results.groupby("Modelo")["F1"].idxmax()
    ].round(4)
)

threshold_results.to_csv(
    RESULTS_DIR / "tabla_12_threshold_cv.csv",
    index=False
)

fig, ax = plt.subplots(figsize=(9, 5))

for model_name in threshold_results["Modelo"].unique():
    temp = threshold_results[
        threshold_results["Modelo"] == model_name
    ]
    ax.plot(
        temp["Threshold"],
        temp["F1"],
        label=model_name
    )
    ax.axvline(
        selected_thresholds[model_name],
        linestyle="--",
        alpha=0.5
    )

ax.set_xlabel("Threshold")
ax.set_ylabel("F1 OOF")
ax.set_title("Selección de threshold mediante Cross-Validation")
ax.legend()
ax.grid(alpha=0.25)

plt.tight_layout()
plt.savefig(RESULTS_DIR / "fig_11_threshold_cv.png", dpi=160)
plt.show()

# 8. Evaluación final — TEST se utiliza por primera vez

In [ ]:
final_models = {
    "Árbol podado": clone(tree_pruned),
    "Random Forest ajustado": clone(rf_tuned),
    "KNN ajustado": clone(knn_final)
}

final_rows = []
final_artifacts = {}

# Baseline
t0 = time.perf_counter()
baseline.fit(X_train, y_train)
baseline_fit_s = time.perf_counter() - t0

t0 = time.perf_counter()
baseline_prob = baseline.predict_proba(X_test)[:, 1]
baseline_pred = baseline.predict(X_test)
baseline_pred_s = time.perf_counter() - t0

bm = classification_metrics(y_test, baseline_pred, baseline_prob)

final_rows.append({
    "Modelo": "Baseline mayoría",
    "Threshold": np.nan,
    "Tiempo_entrenamiento_s": baseline_fit_s,
    "Tiempo_prediccion_s": baseline_pred_s,
    **bm
})

for model_name, model in final_models.items():
    threshold = selected_thresholds[model_name]

    t0 = time.perf_counter()
    model.fit(X_train, y_train)
    fit_s = time.perf_counter() - t0

    t0 = time.perf_counter()
    prob = model.predict_proba(X_test)[:, 1]
    pred_s = time.perf_counter() - t0

    pred = (prob >= threshold).astype(int)

    metrics = classification_metrics(
        y_test,
        pred,
        prob
    )

    final_rows.append({
        "Modelo": model_name,
        "Threshold": threshold,
        "Tiempo_entrenamiento_s": fit_s,
        "Tiempo_prediccion_s": pred_s,
        **metrics
    })

    final_artifacts[model_name] = {
        "model": model,
        "prob": prob,
        "pred": pred,
        "threshold": threshold
    }

final_results = pd.DataFrame(final_rows)

metric_cols = [
    "Accuracy",
    "Balanced_Accuracy",
    "Precision",
    "Recall",
    "Specificity",
    "F1",
    "ROC_AUC",
    "PR_AUC"
]

display(
    final_results.assign(
        **{
            col: (final_results[col] * 100).round(2)
            for col in metric_cols
        }
    )
)

final_results.to_csv(
    RESULTS_DIR / "tabla_13_comparacion_final_test.csv",
    index=False
)

### Cómo leer la tabla final

- **Accuracy:** porcentaje total de clasificaciones correctas.
- **Balanced Accuracy:** promedio del Recall de ambas clases.
- **Precision:** de las reservas predichas como canceladas, cuántas realmente se cancelaron.
- **Recall:** de todas las cancelaciones reales, cuántas detectó el modelo.
- **Specificity:** capacidad de reconocer reservas no canceladas.
- **F1:** equilibrio entre Precision y Recall.
- **ROC-AUC:** capacidad global de ranking entre clases.
- **PR-AUC:** calidad del ranking enfocada en la clase positiva.

## 8.1 Threshold 0.5 vs threshold elegido por CV

In [ ]:
threshold_test_rows = []

for model_name, artifact in final_artifacts.items():
    prob = artifact["prob"]

    for label, threshold in [
        ("Default 0.50", 0.50),
        ("Threshold CV", artifact["threshold"])
    ]:
        pred = (prob >= threshold).astype(int)
        m = classification_metrics(y_test, pred, prob)

        threshold_test_rows.append({
            "Modelo": model_name,
            "Configuracion": label,
            "Threshold": threshold,
            **m
        })

threshold_test_comparison = pd.DataFrame(threshold_test_rows)

display(
    threshold_test_comparison.assign(
        **{
            c: (threshold_test_comparison[c] * 100).round(2)
            for c in metric_cols
        }
    )
)

threshold_test_comparison.to_csv(
    RESULTS_DIR / "tabla_14_threshold_default_vs_cv_test.csv",
    index=False
)

## 8.2 Matrices de confusión — absoluta y normalizada

In [ ]:
for model_name, artifact in final_artifacts.items():
    pred = artifact["pred"]

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))

    cm_abs = confusion_matrix(y_test, pred, labels=[0, 1])
    ConfusionMatrixDisplay(
        cm_abs,
        display_labels=["No cancelada", "Cancelada"]
    ).plot(
        ax=axes[0],
        colorbar=False,
        values_format="d"
    )
    axes[0].set_title(f"{model_name}\nMatriz absoluta")

    cm_norm = confusion_matrix(
        y_test,
        pred,
        labels=[0, 1],
        normalize="true"
    )
    ConfusionMatrixDisplay(
        cm_norm,
        display_labels=["No cancelada", "Cancelada"]
    ).plot(
        ax=axes[1],
        colorbar=False,
        values_format=".1%"
    )
    axes[1].set_title(f"{model_name}\nNormalizada por clase real")

    plt.tight_layout()

    safe_name = (
        model_name.lower()
        .replace(" ", "_")
        .replace("á", "a")
        .replace("ó", "o")
    )

    plt.savefig(
        RESULTS_DIR / f"fig_confusion_{safe_name}.png",
        dpi=160
    )
    plt.show()

## 8.3 Curvas ROC

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 6))

for model_name, artifact in final_artifacts.items():
    prob = artifact["prob"]
    fpr, tpr, _ = roc_curve(y_test, prob)
    auc = roc_auc_score(y_test, prob)

    ax.plot(
        fpr,
        tpr,
        label=f"{model_name} (AUC={auc:.3f})"
    )

ax.plot([0, 1], [0, 1], linestyle="--", label="Azar")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate / Recall")
ax.set_title("Curva ROC — comparación final")
ax.legend()
ax.grid(alpha=0.25)

plt.tight_layout()
plt.savefig(RESULTS_DIR / "fig_12_roc_final.png", dpi=160)
plt.show()

## 8.4 Curvas Precision–Recall

In [ ]:
positive_rate = y_test.mean()

fig, ax = plt.subplots(figsize=(7.5, 6))

for model_name, artifact in final_artifacts.items():
    prob = artifact["prob"]
    precision, recall, _ = precision_recall_curve(y_test, prob)
    ap = average_precision_score(y_test, prob)

    ax.plot(
        recall,
        precision,
        label=f"{model_name} (AP={ap:.3f})"
    )

ax.axhline(
    positive_rate,
    linestyle="--",
    label=f"Prevalencia clase 1={positive_rate:.3f}"
)

ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title("Precision–Recall — comparación final")
ax.legend()
ax.grid(alpha=0.25)

plt.tight_layout()
plt.savefig(RESULTS_DIR / "fig_13_precision_recall_final.png", dpi=160)
plt.show()

## 8.5 Comparación visual de métricas finales

In [ ]:
plot_final = final_results[
    final_results["Modelo"] != "Baseline mayoría"
].copy()

x = np.arange(len(plot_final))
width = 0.18

fig, ax = plt.subplots(figsize=(11, 5.5))

for i, metric in enumerate(
    ["Accuracy", "Balanced_Accuracy", "Recall", "F1"]
):
    ax.bar(
        x + (i - 1.5) * width,
        plot_final[metric],
        width=width,
        label=metric
    )

ax.set_xticks(x)
ax.set_xticklabels(plot_final["Modelo"], rotation=0)
ax.yaxis.set_major_formatter(PercentFormatter(1.0))
ax.set_ylabel("Desempeño")
ax.set_title("Comparación final de modelos sobre TEST")
ax.legend()
ax.grid(axis="y", alpha=0.25)

plt.tight_layout()
plt.savefig(RESULTS_DIR / "fig_14_comparacion_final.png", dpi=160)
plt.show()

# 9. Conclusiones experimentales para el informe

Al finalizar la ejecución, el informe puede organizarse alrededor de estas preguntas:

1. **¿Qué tan difícil es superar el baseline?**  
   Comparar cada modelo contra `DummyClassifier`.

2. **¿Qué atributo tiene mayor poder discriminativo individual?**  
   Usar la tabla de Decision Stumps y contrastarla con la raíz elegida realmente por CART en los 5 folds.

3. **¿El Árbol libre sobreajusta?**  
   Comparar F1 de entrenamiento, F1 de CV, profundidad, hojas y nodos antes/después de la poda.

4. **¿Qué aporta Random Forest?**  
   Comparar RF sin regularización y RF ajustado; discutir estabilidad, `class_weight`, número de árboles y costo computacional.

5. **¿Cuál K funciona mejor para KNN?**  
   Justificarlo mediante Stratified 5-Fold CV, no mediante TEST.

6. **¿Qué efecto tiene ajustar el threshold?**  
   Comparar `0.50` contra el threshold seleccionado con predicciones OOF de TRAIN.

7. **¿Qué modelo maneja mejor ambas clases?**  
   No basarse solo en Accuracy. Comparar Balanced Accuracy, Recall, F1, ROC-AUC y PR-AUC.

8. **¿Qué errores comete cada modelo?**  
   Interpretar matrices de confusión absolutas y normalizadas.

La elección de un modelo final para producción dependería del costo empresarial de los errores. Si perder una cancelación real es más costoso que generar una falsa alerta, se debería priorizar Recall; si ambas clases deben tratarse de manera simétrica, Balanced Accuracy y F1 ofrecen una lectura más equilibrada.

## 10. Archivos generados

In [ ]:
print("Resultados guardados en:")
print(RESULTS_DIR.resolve())

print("\nArchivos:")
for path in sorted(RESULTS_DIR.glob("*")):
    print("-", path.name)